# JSONL boundary investigation

Dead end: str.splitlines() split valid Unicode narrative text into multiple records in the first completed intake run. This notebook reproduces the boundary error using authored text only.


In [ ]:
import json, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'evaluation/registration.json').exists())
sys.path.insert(0, str(ROOT))
import tempfile
from scripts.run_baseline import read
from tests.test_floor_training import rows
from packages.core import Feature
row = rows(1, 2023)[0]
text = "Authored narrative with a Unicode separator\u2028inside its text."
row = row.model_copy(update={"feature": Feature(narrative=text)})
payload = row.model_dump_json() + "\n"
assert len(payload.splitlines()) > 1
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "authored.jsonl"
    path.write_text(payload, encoding="utf-8")
    recovered = read(path)
assert len(recovered) == 1 and recovered[0].feature.narrative == text
intake = json.loads((ROOT/"public_results/intake.json").read_text())
print(json.dumps({"fixture_roundtrip": "passed", "actual_split_counts": intake["counts"]}, indent=2))


The replacement reader uses physical file lines and hides record contents in parse errors. The fixture proves the parsing regression; the aggregate intake file supplies actual population counts.
